# Clustering Track — Gene Expression Cancer RNA-Seq (UCI)
**Goal:** group tumour samples by gene-expression profile with **no labels used during fitting**.
The known cancer types are only used *afterwards* to validate the clusters (rubric §3.3).

Algorithms: **K-Means** (elbow curve) and **Agglomerative Hierarchical** (dendrogram, linkage comparison).
Metrics: Silhouette, Davies-Bouldin, Calinski-Harabasz. Visuals: PCA 2-D (mandatory) and t-SNE.



In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100


Load and Audit

In [ ]:
# The UCI zip (TCGA-PANCAN-HiSeq-801x20531) contains data.csv (samples x genes) and labels.csv (Class).
# Put the unzipped folder anywhere under ../data/ — the loader searches for the two files.
DATA_DIR = Path("../data")
data_path = next(DATA_DIR.rglob("data.csv"), None)
labels_path = next(DATA_DIR.rglob("labels.csv"), None)
if data_path is None:
    raise FileNotFoundError("data.csv not found under ../data/ — unzip the UCI gene-expression download there.")

expr = pd.read_csv(data_path, index_col=0)
labels = pd.read_csv(labels_path, index_col=0)["Class"].loc[expr.index] if labels_path else None

print("Expression matrix (samples x genes):", expr.shape)
print("Missing values:", int(expr.isnull().sum().sum()))
print("Duplicate samples:", int(expr.duplicated().sum()))
print("dtypes:", expr.dtypes.value_counts().to_dict())
if labels is not None:
    print("\nKnown classes (held back — used for validation only):")
    print(labels.value_counts())
